In [1]:
import simulator as sim
import time
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel
from NoisyCircuits.QuantumCircuit import QuantumCircuit as circ
import numpy as np

2026-04-02 10:27:15,672	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
n = 5

In [3]:
def fidelity(p1, p2):
    return np.sum(np.sqrt(p1 * p2))**2

In [4]:
def build_circuit(circuit, n_qubits, depth):
    gate_map = {
        "x": lambda x,p: circuit.X(x),
        "sx": lambda x,p: circuit.SX(x),
        "rz": lambda x,p: circuit.RZ(p[0], x),
        "rx": lambda x,p: circuit.RX(p[0], x)
    }
    np.random.seed(51)
    for _ in range(depth):
        for i in range(n_qubits):
            gate = np.random.choice(list(gate_map.keys()))
            gate_map[gate](i, np.random.uniform(-2*np.pi, 2*np.pi, size=(1,)))
        for i in range(0, n_qubits-1):
            circuit.CZ(i, i+1)

In [5]:
noise_model = CreateNoiseModel("../noise_models/Sample_Noise_Model_Heron_QPU.csv", [["x", "sx", "rz", "rx"], ["cz", "rzz"]]).create_noise_model()

In [6]:
qc = circ(num_qubits=n, noise_model=noise_model, backend_qpu_type="heron", num_trajectories=100, num_cores=10, sim_backend="qulacs", threshold=1e-5)

Completed Extraction of Measurement Errors.
Completed Extraction of two-qubit gate Errors.
Starting post-processing on Single Qubit Errors.


Successfully switched backend to qulacs.


Completed post-processing on Single Qubit Errors.
Processing two-qubit gate errors.
Qubit pair (0, 1): 20/144 errors above threshold (124 filtered out)
Qubit pair (1, 2): 19/144 errors above threshold (125 filtered out)
Qubit pair (1, 0): 20/144 errors above threshold (124 filtered out)
Qubit pair (2, 1): 19/144 errors above threshold (125 filtered out)
Qubit pair (2, 3): 19/144 errors above threshold (125 filtered out)
Qubit pair (3, 2): 19/144 errors above threshold (125 filtered out)
Qubit pair (3, 4): 18/48 errors above threshold (30 filtered out)
Qubit pair (4, 3): 18/48 errors above threshold (30 filtered out)
Qubit pair (0, 1): 20/144 errors above threshold (124 filtered out)
Qubit pair (1, 2): 19/144 errors above threshold (125 filtered out)
Qubit pair (1, 0): 20/144 errors above threshold (124 filtered out)
Qubit pair (2, 1): 19/144 errors above threshold (125 filtered out)
Qubit pair (2, 3): 19/144 errors above threshold (125 filtered out)
Qubit pair (3, 2): 19/144 errors abo

In [7]:
single = {}
double = {}

for qubit in qc.single_qubit_instructions.keys():
    single[qubit] = {}
    for gate in qc.single_qubit_instructions[qubit]:
        single[qubit][gate] = qc.single_qubit_instructions[qubit][gate]["qubit_channel"]
for gate in qc.two_qubit_instructions.keys():
    double[gate] = {}
    for qpair in qc.two_qubit_instructions[gate].keys():
        double[gate][qpair] = qc.two_qubit_instructions[gate][qpair]["qubit_channel"]

In [8]:
from pympler import asizeof

single_reduced = asizeof.asizeof(single)
double_reduced = asizeof.asizeof(double)
single_full = asizeof.asizeof(qc.single_qubit_instructions)
double_full = asizeof.asizeof(qc.two_qubit_instructions)
print(f"Single qubit instructions reduced: {single_reduced/1e6:.2f} MB")
print(f"Two qubit instructions reduced: {double_reduced/1e6:.2f} MB")
print(f"Single qubit instructions full: {single_full/1e6:.2f} MB")
print(f"Two qubit instructions full: {double_full/1e6:.2f} MB")
print(f"Reduction factor single qubit instructions: {single_full/single_reduced:.2f}")
print(f"Reduction factor two qubit instructions: {double_full/double_reduced:.2f}")

Single qubit instructions reduced: 0.06 MB
Two qubit instructions reduced: 0.36 MB
Single qubit instructions full: 0.39 MB
Two qubit instructions full: 1.52 MB
Reduction factor single qubit instructions: 6.02
Reduction factor two qubit instructions: 4.19


In [9]:
build_circuit(qc, n, 100)

In [10]:
instructions = qc.instruction_list

In [ ]:
print("Executing Pure Statevector Simulation:")
t0 = time.perf_counter()
p_ref = qc.run_pure_state(list(range(n)))
t1 = time.perf_counter()

t2 = time.perf_counter()
state = np.zeros(2**n, dtype=np.complex128)
sim.simulate_circuit(instructions, state, single, double, n, False, 1000, 1)
state = state.reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1).astype(np.float64)
t3 = time.perf_counter()

print(f"Fidelity Between Outputs: {fidelity(p_ref, state)}")
print(f"Time taken for Pure State Vector Simulation (Qul): {t1-t0:.5f} seconds")
print(f"Time taken for Pure State Vector Simulation (C++): {t3-t2:.5f} seconds")

Executing Pure Statevector Simulation:
Fidelity Between Outputs: 1.0000000000000013
Time taken for Pure State Vector Simulation (Qul): 0.01488 seconds
Time taken for Pure State Vector Simulation (C++): 0.00781 seconds


In [ ]:
print("Executing Noisy Simulation:")
t0 = time.perf_counter()
p_ref = qc.execute(list(range(n)), 1000)
t1 = time.perf_counter()

m = qc.measurement_error_operator

t2 = time.perf_counter()
state = np.zeros(2**n, dtype=np.complex128)
sim.simulate_circuit(instructions, state, single, double, n, True, 1000, 10)
state /= 1000
state = state.reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1)
state = np.dot(m, state)
t3 = time.perf_counter()

print(f"Fidelity Between Outputs: {fidelity(p_ref, state)}")
print(f"Time taken for Noisy Simulation (Qul): {t1-t0:.5f} seconds")
print(f"Time taken for Noisy Simulation (C++): {t3-t2:.5f} seconds")

Executing Noisy Simulation:
Fidelity Between Outputs: (0.9899261624693241+0j)
Time taken for Noisy Simulation (Qul): 296.28609 seconds
Time taken for Noisy Simulation (C++): 1.75024 seconds


In [13]:
qc.shutdown()